<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/notebooks/03_fair_value_lag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — How fast does Polymarket price BTC information? (main analysis)

Three complementary measurements on the 1-minute panel (0.05–0.95 price band, SE clustered by expiry hour):

1. **Efficiency test** — does BTC information already public at t predict the next Polymarket move?
   Under semi-strong efficiency all coefficients on `d_fair_t … d_fair_{t−5}` are zero.
2. **Impulse response (local projections)** — the share of a fair-value move reflected in the price
   after h minutes. The minutes needed to reach 50% / 90% of the long-run response is the **lag**.
3. **Mispricing half-life** — speed at which the gap between price and fair value closes.

Then the same lag is compared across volatility regimes and time to expiry.

In [ ]:
# Setup — identical in every notebook; works in Colab and locally
import os, sys, subprocess
if "google.colab" in sys.modules:
    REPO = "/content/DSA-Project"
    if os.path.exists(REPO):
        subprocess.run(["git", "-C", REPO, "pull", "-q", "--autostash"])
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/llelus/DSA-Project.git", REPO], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, dataset, analysis
from src.results import save_metrics, savefig

panel = dataset.add_market_diffs(dataset.load_panel())
usable = panel["fair"].notna() & (panel["tau_min"] >= config.MIN_TAU_MIN)
mid = usable & panel["price"].between(config.MID_LOW, config.MID_HIGH)
print(f"Panel: {len(panel):,} market-minutes | {panel.condition_id.nunique():,} markets | "
      f"{panel.market_end.nunique():,} expiry hours")
print(f"Usable rows: {usable.sum():,} | in the {config.MID_LOW}-{config.MID_HIGH} price band: {mid.sum():,}")

## 1. Semi-strong efficiency test

In [ ]:
eff, coefs = analysis.efficiency_test(panel, n_lags=5, mask=mid)
print(f"Joint F = {eff['F']:.2f}, p = {eff['p_value']:.3g}, n = {eff['n']:,}")
print(f"Sum of d_fair coefficients (cumulative delayed response) = {eff['sum_coef']:.3f}")
print(coefs.round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
lags = np.arange(len(coefs))
ax.errorbar(lags, coefs.coef, yerr=1.96 * coefs.se, fmt="o", capsize=4)
ax.axhline(0, color="k", lw=.8)
ax.set_xticks(lags); ax.set_xticklabels([f"d_fair t−{j}" for j in lags])
ax.set_ylabel("Effect on dp t+1"); ax.set_title(f"Efficiency test (joint p = {eff['p_value']:.2g})")
savefig(fig, "lag_efficiency_test"); plt.show()
save_metrics("efficiency_test", **eff, coef={k: float(v) for k, v in coefs.coef.items()})

## 2. Impulse response — the lag in minutes

In [ ]:
lp = analysis.local_projections(panel, config.LP_HORIZONS, mask=mid)
lags_all = analysis.response_lags(lp)
print(lags_all)

b_long = lags_all["b_long"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lp.h, lp.b / b_long, "o-", ms=3, label="b_h / b_long")
ax.fill_between(lp.h, lp.lo / b_long, lp.hi / b_long, alpha=.2, label="95% CI")
for s, ls in [(0.5, "--"), (0.9, ":")]:
    ax.axhline(s, color="gray", ls=ls, lw=1)
ax.set_xlabel("Minutes after the BTC-implied move (h)")
ax.set_ylabel("Share of long-run response")
ax.set_title(f"Polymarket response to BTC information — 50% after {lags_all['lag_50_min']} min, "
             f"90% after {lags_all['lag_90_min']} min")
ax.legend(); savefig(fig, "lag_impulse_response"); plt.show()
save_metrics("impulse_response", **lags_all, horizons=lp.h.tolist(), b=lp.b.tolist(), n_h0=int(lp.n.iloc[0]))

**Reading it:** `b0_share` is the fraction priced within the same minute. If it is close to 1 the market is
effectively real-time at 1-minute resolution; a curve that keeps rising for many minutes is a measurable lag.
Note that rows drop out at long horizons when the market expires, so the sample shifts slightly with h.

## 3. Mispricing half-life (error-correction)

In [ ]:
ecm = analysis.error_correction(panel, mask=mid, demean_gap=True)
ecm_raw = analysis.error_correction(panel, mask=mid, demean_gap=False)
print(f"Within-market gap: beta = {ecm['beta']:.4f} (t = {ecm['t']:.1f}) -> half-life {ecm['half_life_min']:.1f} min")
print(f"Raw gap:           beta = {ecm_raw['beta']:.4f} (t = {ecm_raw['t']:.1f}) -> half-life {ecm_raw['half_life_min']:.1f} min")
save_metrics("error_correction", within=ecm, raw=ecm_raw)

## 4. When is Polymarket slower? (heterogeneity)

In [ ]:
sig_med = panel.loc[mid, "sigma_1m"].median()
groups = {
    "low volatility": mid & (panel.sigma_1m < sig_med),
    "high volatility": mid & (panel.sigma_1m >= sig_med),
    "> 30 min to expiry": mid & (panel.tau_min > 30),
    "≤ 30 min to expiry": mid & (panel.tau_min <= 30),
}
rows, curves = [], {}
for name, m in groups.items():
    lp_g = analysis.local_projections(panel, config.LP_HORIZONS, mask=m)
    curves[name] = lp_g
    rows.append({"group": name, "n": int(m.sum()), **analysis.response_lags(lp_g)})
het = pd.DataFrame(rows).set_index("group")
print(het.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, names in zip(axes, [list(groups)[:2], list(groups)[2:]]):
    for n in names:
        c = curves[n]
        ax.plot(c.h, c.b / c.b.iloc[-1], "o-", ms=3, label=n)
    ax.axhline(0.5, color="gray", ls="--", lw=1); ax.axhline(0.9, color="gray", ls=":", lw=1)
    ax.set_xlabel("Minutes after the BTC-implied move"); ax.legend()
axes[0].set_ylabel("Share of long-run response")
savefig(fig, "lag_heterogeneity"); plt.show()
save_metrics("heterogeneity", **het.to_dict(orient="index"))

## Conclusion template

Fill from `results/metrics.json` after running:

* Efficiency test: joint p = `efficiency_test.p_value` → semi-strong efficiency at 1-minute resolution is
  rejected / not rejected.
* Lag: 50% of a BTC-implied move is priced after `impulse_response.lag_50_min` minutes, 90% after
  `impulse_response.lag_90_min`; `impulse_response.b0_share` is priced in the same minute.
* Mispricing half-life: `error_correction.within.half_life_min` minutes.
* Heterogeneity: compare `heterogeneity.*.lag_50_min` across regimes.

**Caveat:** the fair value is a simple model. A gap can mean Polymarket is slow *or* that it knows
something the model ignores (fat tails, order flow). The impulse response is less sensitive to this
because it measures timing of the response, not its level.

In [ ]:
# Optional: commit outputs back to GitHub (needs Colab secrets GITHUB_TOKEN, GIT_NAME, GIT_EMAIL)
PUSH = False
if PUSH:
    from src.colab import push
    push("results: notebook 03 (fair-value lag)", paths=('figures', 'results'))